# 节点 1：SQLite 与物品 CRUD

这一节点让网页里的物品不再只是临时文字，而是能够保存、修改、删除并在重启后继续存在。

## 1. 本节点目标

把 SQLite 想成家里的物品登记本：页面负责接收操作，Repository 负责翻阅或填写登记本，数据库文件负责长期保存。用户可以添加、查看、编辑和二次确认删除物品。

## 2. 完成结果与验收

- 数据库缺失时自动建立，重复初始化不会清空数据。
- 页面支持物品新增、总览、编辑和二次确认删除。
- 名称会去除首尾空格，空名称和不合理周期会被拒绝。
- 演示数据重复载入不会无限增加。
- 11 项全量测试通过；浏览器实际验证了新增、编辑和删除确认保护。

## 3. 本节点文件结构

```text
app.py                              物品管理页面和友好提示
src/smart_laundry/models.py        物品数据结构与输入规则
src/smart_laundry/database.py      SQLite 连接、事务和建表
src/smart_laundry/repositories.py  参数化 SQL 与 CRUD
src/smart_laundry/demo_data.py     幂等演示数据
scripts/seed_demo_data.py          命令行演示数据入口
tests/conftest.py                  临时测试数据库
tests/test_database.py             建表和约束测试
tests/test_repositories.py         CRUD 与校验测试
tests/test_demo_data.py            重复载入测试
```

## 4. 关键代码解释

源文件：`src/smart_laundry/database.py`

`CREATE TABLE IF NOT EXISTS items` 的意思是：没有表就创建，已经有表就保持不动，因此应用每次重跑都可以安全调用。

`database_connection()` 使用上下文管理器控制连接。正常结束时 `commit()`，发生错误时 `rollback()`，最后无论成功失败都会 `close()`。这避免写入一半和连接长期占用。

源文件：`src/smart_laundry/repositories.py`

SQL 中的 `?` 是参数占位符。物品名称作为参数单独交给 SQLite，不会被误当成 SQL 命令。UI 只调用 Repository，不直接拼 SQL。

In [ ]:
# 这是内存数据库示例，关闭连接后自动消失，不会修改项目数据。
import sqlite3

connection = sqlite3.connect(':memory:')
connection.execute('CREATE TABLE demo (id INTEGER PRIMARY KEY, name TEXT NOT NULL)')
connection.execute('INSERT INTO demo (name) VALUES (?)', ('床单',))
print(connection.execute('SELECT * FROM demo').fetchall())
connection.close()

## 5. 数据流

```mermaid
flowchart LR
    A[用户填写 Streamlit 表单] --> B[models.py 校验与清理]
    B --> C[ItemRepository]
    C --> D[参数化 SQL]
    D --> E[(SQLite 文件)]
    E --> C
    C --> F[Item 对象]
    F --> G[页面表格与成功提示]
```

页面重跑后会再次从 SQLite 读取，因此数据不会只停留在浏览器会话里。

## 6. 关键概念

- **CRUD**：Create 新增、Read 查询、Update 修改、Delete 删除。
- **持久化**：程序关闭后，数据仍保存在文件中。
- **SQLite**：无需单独启动服务器的轻量数据库。
- **主键 ID**：每条记录稳定且唯一的编号。
- **Repository**：集中负责数据读写的代码层。
- **事务**：把一组数据库操作视为一个完整单位。
- **参数化 SQL**：SQL 结构和用户数据分开传递。
- **幂等**：重复执行仍得到稳定结果，不不断制造重复数据。

## 7. 为什么这样设计

首版使用 Python 标准库 `sqlite3`，没有引入 ORM。这样 SQL 行为直接、依赖少，也更容易理解参数化查询和事务。代价是需要手写字段映射；如果后期表关系明显增多，再评估 SQLAlchemy 会更合理。

校验同时放在 Python 入口和数据库约束：前者提供友好中文错误，后者作为最后一道防线，防止其他调用路径写入非法数据。

## 8. 常见错误与排查

1. **数据库无法打开**：先看 `DATABASE_PATH`，再确认父目录是否可写。
2. **database is locked**：关闭正在长期占用数据库的外部工具，稍后重试。
3. **添加后页面没变化**：确认表单提交成功，并检查页面顶部的成功或错误提示。
4. **演示数据重复**：检查是否通过 `seed_demo_data()`，而不是绕过它直接插入。
5. **测试污染正式数据**：测试必须使用 `tmp_path` fixture，不能读取开发 `.env`。
6. **从其他目录启动后路径改变**：配置模块会把相对数据库路径固定解析到项目根目录。

## 9. 面试可能追问

**问：为什么 UI 不直接执行 SQL？**  答：职责分离后，数据操作可以独立测试，也便于未来更换 UI。继续追问可能是 Repository 和 Service 有什么区别。

**问：如何避免 SQL 注入？**  答：所有用户数据通过 `?` 占位符传递，不拼接进 SQL 字符串；测试还用类似 SQL 的物品名称验证表不会被删除。

**问：如何保证重启后数据存在？**  答：数据写入 SQLite 文件，而不是只放在 Streamlit session state。

**问：为什么删除需要二次确认？**  答：删除难以恢复，页面必须先勾选具体物品名称才能启用按钮。

## 10. 必须掌握的最少知识

现在需要掌握：一张表由行和列组成；ID 用来稳定定位某条记录；CRUD 是四种基本操作；页面、Repository、数据库各有不同职责；测试数据库必须与正式数据库隔离。暂时不要求会写复杂 JOIN 或数据库索引。

## 11. 可自测小题

1. CRUD 四个字母分别代表什么？
2. 为什么 `CREATE TABLE IF NOT EXISTS` 可以重复运行？
3. 为什么不能用字符串拼接用户输入和 SQL？
4. `commit` 与 `rollback` 分别在什么时候使用？
5. 为什么测试要使用临时数据库？

<details><summary>查看参考答案</summary>

1. 新增、查询、修改、删除。2. 表存在时不会重新创建或清空。3. 用户文本可能改变 SQL 结构。4. 成功时提交、失败时回滚。5. 避免测试修改开发或正式数据。

</details>

## 12. 动手小练习

1. 运行上面的内存数据库代码，把“床单”改成“窗帘”，观察结果。
2. 在页面新增一件自己的物品，然后重启应用，确认它仍存在。
3. 尝试只输入空格作为名称，观察校验提示；不要修改数据库文件。

## 13. 本节点术语表

| 术语 | 简单解释 |
| --- | --- |
| table | 同一类数据的表格 |
| row | 一条记录 |
| column | 某一种字段 |
| primary key | 唯一定位记录的字段 |
| constraint | 数据库强制执行的规则 |
| transaction | 同时成功或失败的一组操作 |
| fixture | 测试开始前准备的隔离条件 |
| SQL injection | 用户文本被错误当成 SQL 执行的风险 |

## 14. 下一节点连接

本节点的数据层已经被后续状态与天气节点使用：`last_washed_at`、`last_dried_at` 和用户周期会计算天数、超期比例、原因与表情；状态规则位于独立纯函数模块，不交给大模型决定。